# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane: Refresh / Content Opportunity Scoring.**

**My rule in plain words:** a page is worth putting in the refresh review queue if it hasn't
been touched in a while (stale) AND it still gets meaningful real search traffic today
(visible). Staleness alone isn't a reason to review — a stale page nobody sees isn't worth an
editor's time. Traffic alone isn't a reason either — a fresh page doesn't need a refresh review.
It's the combination that defines "an old page still worth someone's attention."

Before coding the rule, I check the two signals it leans on: is "stale" actually associated
with getting worse (the logic behind FlyRank's refresh flags), and is "high search-volume
implies opportunity" (the logic behind FlyRank's quick-win flag) actually true in this slice?

**Signal check 1 — staleness (`days_since_last_update`), behind the refresh flags.**
I bucket by `freshness_tier` and look at the observed down-trend rate per bucket
(`trend_direction` is read here only for validation, never as a rule input).

**Signal check 2 — `search_volume`, behind the quick-win flag.**
Quick-win logic assumes high keyword search-volume means more real traffic upside on the page.
I test that directly against what the page actually gets.

**Reason codes this rule can output:** just one — `stale_but_visible` — since this is a single
AND-rule, not a multi-branch one. (Reasoning behind why only one code, and the two verdicts,
follow in the code cell below.)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")
print(df.shape)

# --- Signal check 1: staleness vs decline rate ---
tier_order = ["0-30", "31-90", "91-180", "181+"]
sig1 = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"),
           pct_down=("trend_direction", lambda s: round((s == "down").mean() * 100, 1)),
           mean_trend_pct=("trend_pct", lambda s: round(s.mean(), 1)))
      .reindex(tier_order)
)
print("\nSignal 1 — staleness vs decline rate:")
print(sig1)
# Reading: down-rate climbs 51.1% -> 58.9% -> 61.1% through 91-180 days, then reverses at 181+
# (47.1%, but n=174 — smallest, noisiest bucket). Real signal through ~180 days, not past it.
# VERDICT: MIXED

# --- Signal check 2: search_volume vs real traffic ---
df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=[-1, 0, 10, 50, 200, 1000, 100000],
    labels=["0", "1-10", "11-50", "51-200", "201-1000", "1000+"],
)
sig2 = (
    df.groupby("volume_bucket", observed=True)
      .agg(n=("content_id", "size"),
           mean_impr_last30=("impressions_last_30d", lambda s: round(s.mean(), 0)),
           mean_clicks_last30=("clicks_last_30d", lambda s: round(s.mean(), 2)),
           mean_ctr_pct=("ctr", lambda s: round(s.mean(), 3)))
)
print("\nSignal 2 — search_volume vs real traffic:")
print(sig2)
print("\ncorrelation, search_volume vs impressions_last_30d:",
      round(df["search_volume"].corr(df["impressions_last_30d"]), 4))
print("correlation, search_volume vs clicks_last_30d:     ",
      round(df["search_volume"].corr(df["clicks_last_30d"]), 4))
# Reading: clicks and CTR actually FALL as search_volume rises (competition effect), and
# correlation with real traffic is ~0. search_volume does not predict opportunity here.
# VERDICT: FALSE — so the rule below uses OBSERVED impressions_last_30d, not search_volume,
# for its "visible" condition.

(30000, 44)

Signal 1 — staleness vs decline rate:
                    n  pct_down  mean_trend_pct
freshness_tier                                 
0-30            20480      51.1             0.8
31-90             175      58.9            -7.4
91-180           9171      61.1           -15.7
181+              174      47.1            -6.8

Signal 2 — search_volume vs real traffic:
                   n  mean_impr_last30  mean_clicks_last30  mean_ctr_pct
volume_bucket                                                           
0              11081            1537.0                5.79         0.438
1-10            7311            1447.0                5.26         0.283
11-50           4989            1575.0                5.21         0.232
51-200          2081            1847.0                5.23         0.192
201-1000        1510            1439.0                3.63         0.170
1000+            560            1872.0                3.27         0.182

correlation, search_volume vs imp

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**The rule (readable on purpose, no fitted weights):**

- `stale` = `days_since_last_update >= 90`
- `visible` = `impressions_last_30d >= 500` (observed traffic, not `search_volume` — signal
  check 2 just showed that's unreliable)
- `score = stale * visible * impressions_last_30d`
- reason code for every scored row: `stale_but_visible`
- action: `review_for_refresh` if `score > 0`, else `no_action`

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_last_30d"] >= 500).astype(int)

df["score"] = stale * visible * df["impressions_last_30d"]
df["reason_code"] = "stale_but_visible"
df["action"] = np.where(df["score"] > 0, "review_for_refresh", "no_action")

n_flagged = (df["score"] > 0).sum()
print(f"flagged for review: {n_flagged} of {len(df)} rows ({n_flagged/len(df)*100:.1f}%)")

base_rate_down = (df["trend_direction"] == "down").mean()
flagged_rate_down = df.loc[df["score"] > 0, "trend_direction"].eq("down").mean()
print(f"base rate of 'down' trend across all rows:      {base_rate_down*100:.1f}%")
print(f"rate of 'down' trend among rows flagged by rule: {flagged_rate_down*100:.1f}%")
print("(trend_direction used ONLY to sanity-check the rule after scoring — never as a rule input)")

out_cols = ["content_id", "client_id", "score", "reason_code", "action"]
queue = df.sort_values("score", ascending=False)[out_cols].reset_index(drop=True)

import os
os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

flagged for review: 3801 of 30000 rows (12.7%)
base rate of 'down' trend across all rows:      54.2%
rate of 'down' trend among rows flagged by rule: 50.6%
(trend_direction used ONLY to sanity-check the rule after scoring — never as a rule input)
wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action
0,content_2dba2b1f9536,client_6208ef0f77,139891,stale_but_visible,review_for_refresh
1,content_4a6607efcb46,client_6208ef0f77,122303,stale_but_visible,review_for_refresh
2,content_5fe46e04994d,client_4e07408562,120791,stale_but_visible,review_for_refresh
3,content_9532f197bbc8,client_4e07408562,109317,stale_but_visible,review_for_refresh
4,content_36ff89c8214e,client_19581e27de,106985,stale_but_visible,review_for_refresh
5,content_2c2606c5d176,client_19581e27de,104248,stale_but_visible,review_for_refresh
6,content_b28d1efd668f,client_6208ef0f77,91655,stale_but_visible,review_for_refresh
7,content_f02b48f88241,client_6208ef0f77,88590,stale_but_visible,review_for_refresh
8,content_6a5b8ccbd700,client_6208ef0f77,78096,stale_but_visible,review_for_refresh
9,content_91652435f57a,client_19581e27de,74135,stale_but_visible,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-10 reviewed below** (the assignment card asks for 10; top-20 is the optional deeper
version, not required here).

Row-by-row (in rank order): action, why it's there, what would make it wrong.

1. **139,891 impr, 104d stale, trend `stable`** — flagged for sheer traffic volume while
   untouched 90+ days. *Wrong if:* it's a reference/evergreen page (glossary, definition page)
   that isn't supposed to change — refreshing would waste editor time for no gain.
2. **122,303 impr, 104d stale, trend `up`** — flagged despite already trending up. *Wrong if:*
   it's improving on its own; a refresh could reset momentum rather than help it.
3. **120,791 impr, 104d stale, trend `down`** — rule working as intended: real traffic, stale,
   already declining. *Wrong if:* the decline is seasonal (yearly-cycle topic) rather than
   content decay — a refresh wouldn't fix a seasonal dip.
4. **109,317 impr, 104d stale, trend `down`, position 2.0** — top-3 position but declining
   anyway. *Wrong if:* the drop is a SERP feature change (snippet/AI overview stealing the
   click) — that's a UX problem, not a staleness problem.
5. **106,985 impr, 104d stale, trend `stable`** — same pattern as #1. *Wrong if:* stability here
   means "already optimal," not "stagnant."
6. **104,248 impr, 104d stale, trend `down`, position 4.2** — good refresh candidate: visible
   traffic, real decline, page-1 position worth protecting. *Wrong if:* the client already has a
   refresh scheduled and this is a duplicate flag.
7. **91,655 impr, 104d stale, trend `stable`, position 26.2** — big traffic number but weak
   position. *Wrong if:* the traffic comes from a different keyword than the one tracked, so a
   refresh aimed at `avg_position` wouldn't move the number that matters.
8. **88,590 impr, 104d stale, trend `up`, position 25.8** — flagged while trending up from a
   weak position. *Wrong if:* it's a new page still climbing naturally — "stale" is catching
   content-age noise, not neglect (worth checking `content_age_days`).
9. **78,096 impr, 104d stale, trend `up`, position 18.3** — same shape as #8; improving-but-
   flagged pages are the pattern most worth a second look before acting.
10. **74,135 impr, 104d stale, trend `stable`, position 7.8** — solid page-1 position, stale,
    flat. *Wrong if:* it's already converting fine and a refresh risks breaking what works.

**Pattern across all 10:** every row shows `days_since_last_update == 104` exactly (see the
code cell below — this isn't a coincidence in my query, the field only has 57 distinct values
in this slice, heavily piled on 20 and 104). So the top-10 ranking is really "top by
`impressions_last_30d` within the 104-day pile" — the rule's ordering power currently comes
almost entirely from traffic volume, not from staleness gradation.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10_cols = ["content_id", "score", "reason_code", "action", "days_since_last_update",
              "impressions_last_30d", "clicks_last_30d", "avg_position", "trend_direction",
              "content_type", "main_intent"]
top10 = df.sort_values("score", ascending=False).head(10)[top10_cols].reset_index(drop=True)
print(top10.to_string())

             content_id   score        reason_code              action  days_since_last_update  impressions_last_30d  clicks_last_30d  avg_position trend_direction     content_type    main_intent
0  content_2dba2b1f9536  139891  stale_but_visible  review_for_refresh                     104                139891              314          27.9          stable  keyword article  informational
1  content_4a6607efcb46  122303  stale_but_visible  review_for_refresh                     104                122303               10           2.2              up  keyword article  informational
2  content_5fe46e04994d  120791  stale_but_visible  review_for_refresh                     104                120791              220           4.2            down  keyword article  informational
3  content_9532f197bbc8  109317  stale_but_visible  review_for_refresh                     104                109317             1176           2.0            down  keyword article  informational
4  content_36ff89c82

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks, named honestly:**

- **Ties dominate the ranking.** `days_since_last_update` has only 57 distinct values in this
  slice, heavily piled on 20 and 104 (8,773 rows sit at exactly 104 days — see the value_counts
  below). Every top-10 row landed on the 104-day pile — the rule can't currently distinguish
  "moderately stale" from "very stale" within that pile; it just sorts by traffic.
- **Rows #2, #8, #9 are flagged despite an `up` trend.** The rule has no way to see recent
  direction — by design, since `trend_direction` is label-derived and correctly excluded from
  the rule. That's the right leakage call, but it does mean the rule keeps flagging pages that
  are already improving on their own, wasting review time.
- **The flagged set isn't enriched for decline.** 50.6% "down" among flagged vs. 54.2% base
  rate (from section 2) — confirming what the top-10 read suggested: this rule finds
  "old + visible," not "old + visible + actually getting worse." A model that properly uses
  trend information (without leaking the future) should beat this by a wide margin.

**Leakage / leave-out check:**
- No future-window columns used as inputs — `trend_direction` / `trend_pct` appear only in
  read-only sanity checks, never multiplied into `score`.
- No FlyRank product flags (its own quick-win/refresh labels) used as inputs — signal check 2
  tested the *logic* behind them, not the flag values themselves.
- `content_id` / `client_id` used for grouping/output only, never as score inputs.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df["days_since_last_update"].value_counts().head(10))
print()
print("distinct values of days_since_last_update:", df["days_since_last_update"].nunique())

days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
14       505
28       466
25       441
7        420
11       246
Name: count, dtype: int64

distinct values of days_since_last_update: 57


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.